# SBA Structure Identification

This notebook loads the prepared numeric SBA dataset after upstream cleaning and featurization have already been completed. It assumes the downstream structure-discovery workflow is operating on numeric features and filters redundant columns using normalized mutual information (NMI) instead of pairwise Spearman correlation.

The analysis then evaluates train/validation/test structure using the `structure_id` package on the training partition.


In [1]:
from pathlib import Path
import pandas as pd
import sys

current_dir = Path.cwd().resolve()
candidate_roots = [current_dir, current_dir.parent] if current_dir.name == 'notebooks' else [current_dir]
project_root = next((root for root in candidate_roots if (root / 'data' / 'model_ready_numeric_data.csv').exists()), current_dir)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

data_path = project_root / 'data' / 'model_ready_numeric_data.csv'
if not data_path.exists():
    raise FileNotFoundError(f'Data file was not found at {data_path}')

df = pd.read_csv(data_path)
print('Columns:', df.columns[:8].tolist(), '...')
partition_col = 'dataset_partition'
print('Partition column:', partition_col)
print(df[[partition_col]].head())
print('\nPartition counts:')
print(df[partition_col].value_counts(dropna=False).to_string())


Columns: ['projectcounty_rcs_te', 'naicscode_rcs_te', 'franchisename_rcs_te', 'grossapproval', 'franchisecode_rcs_te', 'jobssupported', 'projectcounty_rcs', 'naicscode'] ...
Partition column: dataset_partition
  dataset_partition
0             train
1             train
2        validation
3            active
4             train

Partition counts:
dataset_partition
active        55845
train         48396
validation     6050
test           6050


In [2]:
# Use the partition column to select the data split. The prepared numeric dataset already contains
# split membership in a single column, so no duplicated partition columns are required.
partition_col = 'dataset_partition'

train_df = df[df[partition_col] == 'train'].copy()
validation_df = df[df[partition_col] == 'validation'].copy()
test_df = df[df[partition_col] == 'test'].copy()

print(f'train rows: {len(train_df)}')
print(f'validation rows: {len(validation_df)}')
print(f'test rows: {len(test_df)}')

train_df = train_df.drop(columns=[partition_col])
validation_df = validation_df.drop(columns=[partition_col])
test_df = test_df.drop(columns=[partition_col])

print(train_df.head(2).to_string(index=False))


train rows: 48396
validation rows: 6050
test rows: 6050
 projectcounty_rcs_te  naicscode_rcs_te  franchisename_rcs_te  grossapproval  franchisecode_rcs_te  jobssupported projectcounty_rcs  naicscode  borrower_longitude naicscode_rcs  sbadistrictoffice_rcs_te                      businessage_rcs     sbadistrictoffice_rcs  businessage_rcs_te  borrower_latitude  congressionaldistrict_rcs  congressionaldistrict_rcs_te  congressionaldistrict  processingmethod_rcs_te                        franchisename_rcs  loan_status_r
             0.042208          0.036493              0.078640         810000              0.078640              0        MONTGOMERY   721110.0            -89.6499      721110.0                  0.029565 Less than 4 years old but at least 3 ILLINOIS DISTRICT OFFICE             0.022928            39.1793                       13.0                      0.027260                   13.0                 0.034944 CHOICE HOTELS INTERNATIONAL INC.                    0.0
            

In [3]:
from structure_id import structure_id

result = structure_id(
    data=train_df,
    target='loan_status_r',
    task_type='classification',
    structural_covariates=[],
)

print(result.summary())
print('\nSelected columns:')
print(result.selected_columns)
print('\nExplicit column structures:')
for block in result.explicit_column_structures:
    print(block.describe())
    if block.merge_history:
        print('  Merge details:')
        for item in block.merge_history:
            left, right = item["merged_pair"]
            mi = float(item.get("normalized_mutual_information", item.get("mutual_information", 0.0)))
            print(f"    {left} <-> {right}: normalized MI = {mi:.6f}, distance={1.0 - mi:.6f}")
        print('\n  Mutual information matrix:')
        print(pd.DataFrame(block.correlation_matrix, index=block.cluster_members, columns=block.cluster_members))


Structure Identification Summary
Target: loan_status_r
Task type: classification
Provisional candidates: 16
Selected candidates: 17
Explicit column blocks: 0
Explicit row clusters: 159
Instance structure summary:
  noise=15660
  cluster_0=45
  cluster_1=55
  cluster_2=141
  cluster_3=144
  cluster_4=254
  cluster_5=299
  cluster_6=721
  cluster_7=30
  cluster_8=30
  cluster_9=187
  cluster_10=141
  cluster_11=4296
  cluster_12=2623
  cluster_13=25
  cluster_14=20
  cluster_15=40
  cluster_16=27
  cluster_17=12131
  cluster_18=135
  cluster_19=35
  cluster_20=182
  cluster_21=47
  cluster_22=56
  cluster_23=50
  cluster_24=34
  cluster_25=18
  cluster_26=30
  cluster_27=70
  cluster_28=35
  cluster_29=23
  cluster_30=15
  cluster_31=40
  cluster_32=52
  cluster_33=24
  cluster_34=23
  cluster_35=138
  cluster_36=16
  cluster_37=15
  cluster_38=51
  cluster_39=52
  cluster_40=36
  cluster_41=15
  cluster_42=24
  cluster_43=40
  cluster_44=34
  cluster_45=21
  cluster_46=16
  cluster_47=2

In [4]:
from IPython.display import Markdown, display
from structure_id.reporting import export_cluster_profile_report

report_dir = project_root / "reports"
exports = export_cluster_profile_report(
    result=result,
    output_dir=report_dir,
    filename_prefix="training_cluster_profile",
)

print(f"Saved markdown report: {exports['markdown']}")
print(f"Saved HTML report: {exports['html']}")
display(Markdown(exports["markdown"].read_text(encoding="utf-8")))


Saved markdown report: /home/rajiv/programming/structure_identification_for_CS_data/reports/training_cluster_profile.md
Saved HTML report: /home/rajiv/programming/structure_identification_for_CS_data/reports/training_cluster_profile.html


# Training Cluster Profile

## Summary
- Noise rows: 15660
- Valid clusters: 159

## Cluster-level interpretation
### Cluster 0
- Size: 45
- Leading features: borrower_longitude (+0.43sd), businessage_rcs_te (-0.38sd), processingmethod_rcs_te (+0.37sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 1
- Size: 55
- Leading features: businessage_rcs_te (-3.39sd), congressionaldistrict_rcs_te (-0.19sd), naicscode (+0.19sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 10
- Size: 141
- Leading features: borrower_latitude (-5.68sd), borrower_longitude (+4.83sd), sbadistrictoffice_rcs_te (+3.83sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 100
- Size: 16
- Leading features: congressionaldistrict_rcs (+2.26sd), congressionaldistrict (+2.26sd), congressionaldistrict_rcs_te (-1.67sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 101
- Size: 32
- Leading features: congressionaldistrict_rcs (+2.39sd), congressionaldistrict (+2.39sd), congressionaldistrict_rcs_te (-2.25sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 102
- Size: 46
- Leading features: congressionaldistrict_rcs (+2.38sd), congressionaldistrict (+2.38sd), congressionaldistrict_rcs_te (-2.28sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 103
- Size: 41
- Leading features: sbadistrictoffice_rcs_te (+1.26sd), projectcounty_rcs_te (-0.75sd), processingmethod_rcs_te (-0.66sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 104
- Size: 24
- Leading features: congressionaldistrict_rcs_te (+1.18sd), naicscode_rcs_te (-0.74sd), borrower_longitude (+0.66sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 105
- Size: 18
- Leading features: naicscode (-1.46sd), projectcounty_rcs_te (+0.66sd), processingmethod_rcs_te (-0.66sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 106
- Size: 17
- Leading features: naicscode (+1.26sd), borrower_longitude (+1.26sd), projectcounty_rcs_te (-1.05sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 107
- Size: 24
- Leading features: processingmethod_rcs_te (-0.66sd), naicscode_rcs_te (-0.63sd), borrower_longitude (+0.58sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 108
- Size: 27
- Leading features: naicscode_rcs_te (+1.86sd), borrower_latitude (+1.14sd), naicscode (+1.09sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 109
- Size: 93
- Leading features: borrower_longitude (-1.40sd), sbadistrictoffice_rcs_te (-0.76sd), processingmethod_rcs_te (-0.66sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 11
- Size: 4296
- Leading features: businessage_rcs_te (+2.91sd), naicscode (+0.47sd), naicscode_rcs_te (+0.40sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 110
- Size: 38
- Leading features: borrower_latitude (+1.01sd), projectcounty_rcs_te (-0.76sd), congressionaldistrict_rcs_te (+0.72sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 111
- Size: 21
- Leading features: borrower_longitude (+0.87sd), projectcounty_rcs_te (+0.83sd), processingmethod_rcs_te (-0.66sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 112
- Size: 17
- Leading features: naicscode_rcs_te (-1.17sd), processingmethod_rcs_te (-0.66sd), borrower_longitude (+0.57sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 113
- Size: 47
- Leading features: naicscode_rcs_te (+1.86sd), naicscode (+1.09sd), borrower_longitude (+1.03sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 114
- Size: 32
- Leading features: borrower_latitude (-1.10sd), borrower_longitude (+0.66sd), processingmethod_rcs_te (-0.66sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 115
- Size: 41
- Leading features: borrower_latitude (-1.33sd), borrower_longitude (+0.68sd), processingmethod_rcs_te (-0.66sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 116
- Size: 64
- Leading features: borrower_longitude (+1.18sd), congressionaldistrict_rcs_te (+1.07sd), borrower_latitude (+0.71sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 117
- Size: 28
- Leading features: naicscode_rcs_te (+1.76sd), naicscode (+0.92sd), borrower_longitude (-0.79sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 118
- Size: 48
- Leading features: processingmethod_rcs_te (-0.66sd), grossapproval (-0.60sd), naicscode_rcs_te (-0.58sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 119
- Size: 23
- Leading features: borrower_longitude (-0.87sd), projectcounty_rcs_te (-0.78sd), sbadistrictoffice_rcs_te (-0.72sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 12
- Size: 2623
- Leading features: businessage_rcs_te (-1.41sd), naicscode (+0.57sd), grossapproval (+0.53sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 120
- Size: 60
- Leading features: borrower_longitude (-0.88sd), projectcounty_rcs_te (-0.76sd), sbadistrictoffice_rcs_te (-0.72sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 121
- Size: 23
- Leading features: naicscode (-1.14sd), naicscode_rcs_te (-0.84sd), borrower_latitude (+0.68sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 122
- Size: 27
- Leading features: borrower_longitude (-1.42sd), borrower_latitude (+1.20sd), sbadistrictoffice_rcs_te (-1.00sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 123
- Size: 60
- Leading features: naicscode (+1.57sd), borrower_latitude (+0.81sd), congressionaldistrict_rcs (-0.67sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 124
- Size: 18
- Leading features: naicscode (-1.69sd), borrower_longitude (-1.39sd), projectcounty_rcs_te (-0.80sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 125
- Size: 22
- Leading features: naicscode (+1.60sd), projectcounty_rcs_te (-1.11sd), borrower_longitude (-0.99sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 126
- Size: 28
- Leading features: naicscode (-1.39sd), borrower_latitude (+0.67sd), processingmethod_rcs_te (-0.66sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 127
- Size: 115
- Leading features: borrower_longitude (+1.18sd), borrower_latitude (+0.91sd), congressionaldistrict_rcs (-0.77sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 128
- Size: 22
- Leading features: naicscode (-1.63sd), borrower_longitude (-0.88sd), sbadistrictoffice_rcs_te (-0.74sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 129
- Size: 19
- Leading features: naicscode (-1.40sd), borrower_longitude (+1.19sd), projectcounty_rcs_te (+1.04sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 13
- Size: 25
- Leading features: franchisecode_rcs_te (+4.60sd), franchisename_rcs_te (+4.60sd), grossapproval (+0.90sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 130
- Size: 24
- Leading features: naicscode (+1.40sd), congressionaldistrict_rcs (-0.73sd), congressionaldistrict (-0.73sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 131
- Size: 23
- Leading features: naicscode (-1.69sd), borrower_longitude (+1.20sd), borrower_latitude (+0.79sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 132
- Size: 20
- Leading features: congressionaldistrict_rcs (-0.71sd), congressionaldistrict (-0.71sd), projectcounty_rcs_te (+0.70sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 133
- Size: 23
- Leading features: naicscode (-1.54sd), borrower_longitude (+1.18sd), borrower_latitude (+0.72sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 134
- Size: 26
- Leading features: naicscode (+1.60sd), borrower_longitude (+1.19sd), borrower_latitude (+0.71sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 135
- Size: 17
- Leading features: naicscode_rcs_te (-0.97sd), borrower_longitude (+0.87sd), congressionaldistrict_rcs (-0.70sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 136
- Size: 28
- Leading features: borrower_longitude (+0.78sd), congressionaldistrict_rcs (-0.68sd), congressionaldistrict (-0.68sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 137
- Size: 40
- Leading features: processingmethod_rcs_te (-0.66sd), naicscode_rcs_te (-0.64sd), grossapproval (-0.62sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 138
- Size: 21
- Leading features: borrower_longitude (+1.12sd), projectcounty_rcs_te (-0.66sd), processingmethod_rcs_te (-0.66sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 139
- Size: 16
- Leading features: borrower_longitude (+1.19sd), projectcounty_rcs_te (+1.07sd), borrower_latitude (+0.72sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 14
- Size: 20
- Leading features: borrower_longitude (-3.15sd), projectcounty_rcs_te (+2.56sd), borrower_latitude (-2.51sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 140
- Size: 16
- Leading features: borrower_longitude (+1.18sd), projectcounty_rcs_te (+0.85sd), borrower_latitude (+0.72sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 141
- Size: 15
- Leading features: naicscode (-1.68sd), borrower_longitude (-1.36sd), sbadistrictoffice_rcs_te (-0.78sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 142
- Size: 51
- Leading features: borrower_longitude (+1.19sd), borrower_latitude (+0.72sd), processingmethod_rcs_te (-0.66sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 143
- Size: 49
- Leading features: borrower_longitude (+1.18sd), naicscode_rcs_te (-0.73sd), borrower_latitude (+0.70sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 144
- Size: 31
- Leading features: borrower_longitude (-1.33sd), naicscode_rcs_te (-0.84sd), sbadistrictoffice_rcs_te (-0.76sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 145
- Size: 47
- Leading features: naicscode (-1.38sd), borrower_latitude (+1.06sd), projectcounty_rcs_te (-0.84sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 146
- Size: 55
- Leading features: borrower_longitude (-1.40sd), projectcounty_rcs_te (-0.80sd), sbadistrictoffice_rcs_te (-0.76sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 147
- Size: 57
- Leading features: borrower_longitude (-1.36sd), sbadistrictoffice_rcs_te (-0.78sd), naicscode_rcs_te (-0.67sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 148
- Size: 44
- Leading features: projectcounty_rcs_te (-0.93sd), congressionaldistrict_rcs (-0.76sd), congressionaldistrict (-0.76sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 149
- Size: 18
- Leading features: naicscode (-1.36sd), projectcounty_rcs_te (-0.95sd), borrower_longitude (-0.88sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 15
- Size: 40
- Leading features: borrower_longitude (-3.20sd), borrower_latitude (-2.49sd), projectcounty_rcs_te (-1.16sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 150
- Size: 15
- Leading features: borrower_longitude (-0.86sd), sbadistrictoffice_rcs_te (-0.76sd), processingmethod_rcs_te (-0.66sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 151
- Size: 45
- Leading features: borrower_longitude (-0.86sd), naicscode_rcs_te (-0.80sd), sbadistrictoffice_rcs_te (-0.76sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 152
- Size: 29
- Leading features: naicscode_rcs_te (-0.79sd), congressionaldistrict_rcs (-0.76sd), congressionaldistrict (-0.76sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 153
- Size: 108
- Leading features: borrower_latitude (+1.02sd), projectcounty_rcs_te (-0.73sd), naicscode_rcs_te (-0.70sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 154
- Size: 81
- Leading features: borrower_longitude (-1.04sd), congressionaldistrict_rcs (-0.71sd), congressionaldistrict (-0.71sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 155
- Size: 69
- Leading features: borrower_longitude (-1.36sd), projectcounty_rcs_te (-0.82sd), sbadistrictoffice_rcs_te (-0.78sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 156
- Size: 110
- Leading features: projectcounty_rcs_te (-0.99sd), borrower_longitude (-0.91sd), sbadistrictoffice_rcs_te (-0.72sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 157
- Size: 74
- Leading features: projectcounty_rcs_te (-0.88sd), borrower_longitude (-0.87sd), naicscode_rcs_te (-0.76sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 158
- Size: 47
- Leading features: borrower_longitude (-1.24sd), projectcounty_rcs_te (-1.17sd), borrower_latitude (+1.04sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 16
- Size: 27
- Leading features: borrower_latitude (+3.63sd), borrower_longitude (-2.74sd), congressionaldistrict_rcs_te (+2.51sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 17
- Size: 12131
- Leading features: grossapproval (+0.19sd), processingmethod_rcs_te (-0.19sd), jobssupported (-0.12sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 18
- Size: 135
- Leading features: borrower_longitude (-3.22sd), borrower_latitude (-2.46sd), processingmethod_rcs_te (+1.44sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 19
- Size: 35
- Leading features: franchisecode_rcs_te (+9.26sd), franchisename_rcs_te (+9.26sd), naicscode (+0.94sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 2
- Size: 141
- Leading features: businessage_rcs_te (+2.41sd), naicscode (+0.49sd), processingmethod_rcs_te (+0.39sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 20
- Size: 182
- Leading features: franchisecode_rcs_te (+6.00sd), franchisename_rcs_te (+5.99sd), congressionaldistrict_rcs (-0.49sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 21
- Size: 47
- Leading features: sbadistrictoffice_rcs_te (+4.86sd), congressionaldistrict_rcs (-0.68sd), congressionaldistrict (-0.68sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 22
- Size: 56
- Leading features: sbadistrictoffice_rcs_te (+3.50sd), borrower_latitude (-1.09sd), processingmethod_rcs_te (+0.83sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 23
- Size: 50
- Leading features: borrower_latitude (+3.55sd), borrower_longitude (-2.74sd), congressionaldistrict_rcs_te (+2.51sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 24
- Size: 34
- Leading features: congressionaldistrict_rcs_te (+3.34sd), congressionaldistrict_rcs (+2.75sd), congressionaldistrict (+2.75sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 25
- Size: 18
- Leading features: congressionaldistrict_rcs (+3.61sd), congressionaldistrict (+3.61sd), congressionaldistrict_rcs_te (+2.47sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 26
- Size: 30
- Leading features: congressionaldistrict_rcs_te (+3.61sd), congressionaldistrict_rcs (+2.51sd), congressionaldistrict (+2.51sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 27
- Size: 70
- Leading features: congressionaldistrict_rcs_te (+3.05sd), congressionaldistrict_rcs (+3.02sd), congressionaldistrict (+3.02sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 28
- Size: 35
- Leading features: sbadistrictoffice_rcs_te (+4.68sd), projectcounty_rcs_te (+3.45sd), congressionaldistrict_rcs_te (+1.38sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 29
- Size: 23
- Leading features: projectcounty_rcs_te (+3.82sd), congressionaldistrict_rcs_te (+2.51sd), borrower_latitude (+1.18sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 3
- Size: 144
- Leading features: businessage_rcs_te (+3.25sd), naicscode (+0.61sd), jobssupported (+0.26sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 30
- Size: 15
- Leading features: congressionaldistrict_rcs_te (+2.51sd), processingmethod_rcs_te (+1.52sd), borrower_latitude (+1.35sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 31
- Size: 40
- Leading features: congressionaldistrict_rcs_te (+2.51sd), processingmethod_rcs_te (+1.46sd), borrower_latitude (+1.18sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 32
- Size: 52
- Leading features: congressionaldistrict_rcs_te (+2.51sd), borrower_longitude (+1.06sd), congressionaldistrict_rcs (-0.87sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 33
- Size: 24
- Leading features: congressionaldistrict_rcs_te (+2.51sd), projectcounty_rcs_te (+1.95sd), borrower_latitude (+0.95sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 34
- Size: 23
- Leading features: congressionaldistrict_rcs_te (+2.51sd), borrower_latitude (+1.44sd), sbadistrictoffice_rcs_te (-1.06sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 35
- Size: 138
- Leading features: congressionaldistrict_rcs_te (+2.51sd), borrower_latitude (+1.13sd), congressionaldistrict_rcs (-0.87sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 36
- Size: 16
- Leading features: sbadistrictoffice_rcs_te (+3.19sd), congressionaldistrict_rcs_te (-2.07sd), congressionaldistrict_rcs (+1.16sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 37
- Size: 15
- Leading features: franchisecode_rcs_te (+3.12sd), franchisename_rcs_te (+3.12sd), naicscode (+1.08sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 38
- Size: 51
- Leading features: congressionaldistrict_rcs_te (-4.18sd), congressionaldistrict_rcs (+3.19sd), congressionaldistrict (+3.19sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 39
- Size: 52
- Leading features: naicscode (-3.09sd), naicscode_rcs_te (+2.68sd), borrower_longitude (+0.96sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 4
- Size: 254
- Leading features: businessage_rcs_te (-1.32sd), grossapproval (+0.46sd), naicscode (+0.44sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 40
- Size: 36
- Leading features: naicscode (-3.09sd), naicscode_rcs_te (+2.68sd), processingmethod_rcs_te (+1.43sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 41
- Size: 15
- Leading features: projectcounty_rcs_te (+1.96sd), borrower_longitude (-1.38sd), congressionaldistrict_rcs (+0.82sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 42
- Size: 24
- Leading features: franchisecode_rcs_te (-4.20sd), franchisename_rcs_te (-4.20sd), naicscode (+1.09sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 43
- Size: 40
- Leading features: sbadistrictoffice_rcs_te (+2.02sd), processingmethod_rcs_te (+1.50sd), congressionaldistrict_rcs_te (-1.24sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 44
- Size: 34
- Leading features: congressionaldistrict_rcs_te (+2.79sd), borrower_latitude (-1.69sd), congressionaldistrict_rcs (+0.99sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 45
- Size: 21
- Leading features: congressionaldistrict_rcs_te (+2.79sd), borrower_latitude (-1.70sd), processingmethod_rcs_te (+1.52sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 46
- Size: 16
- Leading features: congressionaldistrict_rcs_te (+2.79sd), borrower_longitude (-1.26sd), congressionaldistrict_rcs (+0.99sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 47
- Size: 21
- Leading features: sbadistrictoffice_rcs_te (+3.13sd), projectcounty_rcs_te (+1.12sd), borrower_longitude (+1.05sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 48
- Size: 20
- Leading features: projectcounty_rcs_te (+3.61sd), sbadistrictoffice_rcs_te (+2.84sd), processingmethod_rcs_te (+1.48sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 49
- Size: 18
- Leading features: projectcounty_rcs_te (+1.16sd), sbadistrictoffice_rcs_te (+0.81sd), congressionaldistrict_rcs (+0.74sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 5
- Size: 299
- Leading features: grossapproval (+0.72sd), businessage_rcs_te (+0.68sd), naicscode (+0.50sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 50
- Size: 15
- Leading features: projectcounty_rcs_te (+2.87sd), sbadistrictoffice_rcs_te (+1.56sd), borrower_longitude (+1.07sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 51
- Size: 21
- Leading features: projectcounty_rcs_te (+3.24sd), sbadistrictoffice_rcs_te (+1.22sd), borrower_longitude (+1.19sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 52
- Size: 25
- Leading features: projectcounty_rcs_te (+1.54sd), borrower_latitude (-1.18sd), congressionaldistrict_rcs_te (+0.99sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 53
- Size: 27
- Leading features: congressionaldistrict_rcs_te (+1.14sd), borrower_latitude (-1.08sd), projectcounty_rcs_te (-0.84sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 54
- Size: 38
- Leading features: processingmethod_rcs_te (+1.52sd), projectcounty_rcs_te (+1.45sd), congressionaldistrict_rcs_te (+1.08sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 55
- Size: 49
- Leading features: congressionaldistrict_rcs_te (-1.89sd), processingmethod_rcs_te (+1.52sd), congressionaldistrict_rcs (+1.31sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 56
- Size: 5250
- Leading features: processingmethod_rcs_te (+1.41sd), grossapproval (-0.40sd), businessage_rcs_te (-0.31sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 57
- Size: 75
- Leading features: congressionaldistrict_rcs (+2.80sd), congressionaldistrict (+2.80sd), borrower_longitude (-1.20sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 58
- Size: 21
- Leading features: projectcounty_rcs_te (+2.02sd), borrower_longitude (-0.83sd), borrower_latitude (-0.79sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 59
- Size: 46
- Leading features: processingmethod_rcs_te (-1.71sd), congressionaldistrict_rcs (-0.69sd), congressionaldistrict (-0.69sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 6
- Size: 721
- Leading features: businessage_rcs_te (+0.35sd), processingmethod_rcs_te (+0.28sd), naicscode (-0.19sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 60
- Size: 71
- Leading features: congressionaldistrict_rcs_te (-2.07sd), borrower_latitude (-1.76sd), congressionaldistrict_rcs (+1.16sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 61
- Size: 33
- Leading features: borrower_latitude (+1.49sd), borrower_longitude (-1.40sd), congressionaldistrict_rcs_te (+0.76sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 62
- Size: 73
- Leading features: congressionaldistrict_rcs_te (-3.47sd), congressionaldistrict_rcs (+2.09sd), congressionaldistrict (+2.09sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 63
- Size: 26
- Leading features: congressionaldistrict_rcs (+2.17sd), congressionaldistrict (+2.17sd), congressionaldistrict_rcs_te (+1.84sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 64
- Size: 35
- Leading features: congressionaldistrict_rcs (+2.01sd), congressionaldistrict (+2.01sd), borrower_longitude (-1.20sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 65
- Size: 23
- Leading features: congressionaldistrict_rcs (+2.01sd), congressionaldistrict (+2.01sd), naicscode_rcs_te (-1.25sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 66
- Size: 58
- Leading features: congressionaldistrict_rcs_te (-2.07sd), borrower_longitude (-1.28sd), congressionaldistrict_rcs (+1.16sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 67
- Size: 25
- Leading features: borrower_longitude (-1.36sd), congressionaldistrict_rcs_te (+1.32sd), sbadistrictoffice_rcs_te (-0.78sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 68
- Size: 18
- Leading features: borrower_latitude (-1.78sd), congressionaldistrict_rcs (+1.41sd), congressionaldistrict (+1.41sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 69
- Size: 21
- Leading features: borrower_latitude (-1.60sd), naicscode (+1.58sd), sbadistrictoffice_rcs_te (-0.95sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 7
- Size: 30
- Leading features: borrower_latitude (-5.68sd), borrower_longitude (+4.83sd), sbadistrictoffice_rcs_te (+3.83sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 70
- Size: 24
- Leading features: sbadistrictoffice_rcs_te (+1.56sd), borrower_longitude (+1.06sd), processingmethod_rcs_te (-0.66sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 71
- Size: 213
- Leading features: sbadistrictoffice_rcs_te (+1.68sd), projectcounty_rcs_te (+1.08sd), borrower_longitude (+1.08sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 72
- Size: 42
- Leading features: borrower_latitude (+1.57sd), borrower_longitude (-1.41sd), projectcounty_rcs_te (+1.10sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 73
- Size: 29
- Leading features: borrower_latitude (-1.41sd), projectcounty_rcs_te (-1.28sd), sbadistrictoffice_rcs_te (-0.95sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 74
- Size: 15
- Leading features: borrower_latitude (-1.79sd), congressionaldistrict_rcs_te (+1.58sd), congressionaldistrict_rcs (+1.33sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 75
- Size: 39
- Leading features: borrower_latitude (-1.62sd), congressionaldistrict_rcs_te (+0.95sd), sbadistrictoffice_rcs_te (-0.95sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 76
- Size: 39
- Leading features: borrower_latitude (-1.68sd), projectcounty_rcs_te (-1.05sd), sbadistrictoffice_rcs_te (-0.95sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 77
- Size: 57
- Leading features: borrower_latitude (-1.76sd), congressionaldistrict_rcs (+1.24sd), congressionaldistrict (+1.24sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 78
- Size: 15
- Leading features: naicscode (-1.64sd), borrower_latitude (-1.44sd), sbadistrictoffice_rcs_te (-0.95sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 79
- Size: 126
- Leading features: borrower_latitude (-1.45sd), sbadistrictoffice_rcs_te (-0.95sd), processingmethod_rcs_te (-0.66sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 8
- Size: 30
- Leading features: borrower_latitude (-5.68sd), borrower_longitude (+4.83sd), sbadistrictoffice_rcs_te (+3.83sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 80
- Size: 38
- Leading features: congressionaldistrict_rcs (-0.76sd), congressionaldistrict (-0.76sd), sbadistrictoffice_rcs_te (+0.66sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 81
- Size: 43
- Leading features: congressionaldistrict_rcs_te (+1.58sd), congressionaldistrict_rcs (+1.33sd), congressionaldistrict (+1.33sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 82
- Size: 17
- Leading features: naicscode_rcs_te (+1.86sd), borrower_longitude (-1.40sd), naicscode (+1.09sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 83
- Size: 31
- Leading features: borrower_latitude (-1.09sd), projectcounty_rcs_te (-1.05sd), processingmethod_rcs_te (-0.66sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 84
- Size: 43
- Leading features: borrower_longitude (-1.24sd), congressionaldistrict_rcs_te (+1.18sd), congressionaldistrict_rcs (+0.91sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 85
- Size: 166
- Leading features: borrower_longitude (-1.39sd), congressionaldistrict_rcs_te (+1.05sd), sbadistrictoffice_rcs_te (-0.76sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 86
- Size: 78
- Leading features: sbadistrictoffice_rcs_te (-1.79sd), borrower_latitude (+1.48sd), borrower_longitude (-1.18sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 87
- Size: 31
- Leading features: borrower_latitude (+1.52sd), borrower_longitude (-1.41sd), processingmethod_rcs_te (-0.66sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 88
- Size: 87
- Leading features: congressionaldistrict_rcs (+2.64sd), congressionaldistrict (+2.64sd), borrower_longitude (-1.15sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 89
- Size: 120
- Leading features: congressionaldistrict_rcs (+1.46sd), congressionaldistrict (+1.46sd), borrower_longitude (-1.19sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 9
- Size: 187
- Leading features: borrower_latitude (-5.68sd), borrower_longitude (+4.83sd), sbadistrictoffice_rcs_te (+3.83sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 90
- Size: 48
- Leading features: congressionaldistrict_rcs_te (-2.61sd), congressionaldistrict_rcs (+1.75sd), congressionaldistrict (+1.75sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 91
- Size: 21
- Leading features: congressionaldistrict_rcs (+1.25sd), congressionaldistrict (+1.25sd), borrower_longitude (-1.21sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 92
- Size: 88
- Leading features: congressionaldistrict_rcs (+1.49sd), congressionaldistrict (+1.49sd), borrower_longitude (-1.20sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 93
- Size: 163
- Leading features: congressionaldistrict_rcs (+3.32sd), congressionaldistrict (+3.32sd), borrower_longitude (-1.15sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 94
- Size: 20
- Leading features: congressionaldistrict_rcs (+3.36sd), congressionaldistrict (+3.36sd), congressionaldistrict_rcs_te (-2.62sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 95
- Size: 54
- Leading features: congressionaldistrict_rcs (+1.80sd), congressionaldistrict (+1.80sd), congressionaldistrict_rcs_te (-1.37sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 96
- Size: 32
- Leading features: congressionaldistrict_rcs (+3.43sd), congressionaldistrict (+3.43sd), congressionaldistrict_rcs_te (-1.91sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 97
- Size: 131
- Leading features: congressionaldistrict_rcs (+2.95sd), congressionaldistrict (+2.95sd), congressionaldistrict_rcs_te (-2.05sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 98
- Size: 33
- Leading features: borrower_latitude (+1.66sd), borrower_longitude (-1.41sd), congressionaldistrict_rcs (-0.78sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 99
- Size: 58
- Leading features: congressionaldistrict_rcs_te (-2.24sd), congressionaldistrict_rcs (+1.92sd), congressionaldistrict (+1.92sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.
